In [16]:
import pymupdf4llm
import sys
import os
from pathlib import Path
sys.path.append(str(Path().resolve().parent))
from controllers.utils import extract_text, clean_markdown

In [17]:
path = os.path.join(os.getcwd(), "../docs/simCLR.pdf")
print(path)

/Users/ayushjaswal/Developer/personal-projects/groundedQAproject/notebooks/../docs/simCLR.pdf


In [18]:
mkdn = clean_markdown(extract_text(path))

In [ ]:
# from IPython.display import display, Markdown

# display(Markdown(mkdn))

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter, Language

header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "title"),
        ("##", "section"),
        ("###", "subsection"),
    ]
)

header_chunks = header_splitter.split_text(mkdn)

char_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

final_chunks = char_splitter.split_documents(header_chunks)
print(f"Total chunks: {len(final_chunks)}")
print("\n--- Chunk 0 ---")
print(final_chunks[0].page_content)
print(final_chunks[0].metadata)

/Users/ayushjaswal/Developer/personal-projects/groundedQAproject/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total chunks: 106

--- Chunk 0 ---
**A Simple Framework for Contrastive Learning of Visual Representations**  
**Ting Chen**[1] **Simon Kornblith**[1] **Mohammad Norouzi**[1] **Geoffrey Hinton**[1]
{}


In [ ]:
len(final_chunks)

106

In [ ]:
from sentence_transformers import SentenceTransformer

embeddings_model = SentenceTransformer("all-MiniLM-L6-v2")
print(embeddings_model)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4375.17it/s]


SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)


In [ ]:
texts = [chunk.page_content for chunk in final_chunks]

embeddings = embeddings_model.encode(texts, show_progress_bar=True)


print(f"Chunks: {len(texts)}")
print(f"Embedding shape: {embeddings.shape}") 

Batches: 100%|██████████| 4/4 [00:00<00:00,  4.29it/s]

Chunks: 106
Embedding shape: (106, 384)


In [ ]:
embeddings_list = embeddings.tolist()
type(embeddings_list), type(embeddings_list[0])

(list, list)

In [7]:
import chromadb

chroma_client = chromadb.PersistentClient(path="../knowledgebase")

# metadatas = []
# for i, chunk in enumerate(final_chunks):
#     meta = chunk.metadata.copy() 
#     meta["chunk_id"] = i
#     meta["source"] = "simCLR.pdf"
#     metadatas.append(meta)

# reindex
# chroma_client.delete_collection(name="knowledgebase")
collection = chroma_client.get_or_create_collection(name="knowledgebase")

# collection.add(
#     embeddings=embeddings.tolist(),
#     documents=texts,
#     ids=[str(i) for i in range(len(texts))],
#     metadatas=metadatas
# )

# print(f"Stored {collection.count()} chunks")

In [ ]:
#print(collection.get('160', include=['embeddings', 'documents', 'metadatas']))

In [ ]:
query = "what is sim?"

results = collection.query(
    query_texts=[query],
    n_results=4,
    include=["documents", "distances", "metadatas"]
)

In [ ]:
results

{'ids': [['8', '100', '7', '5']],
 'embeddings': None,
 'documents': [['Inspired by recent contrastive learning algorithms (see Section 7 for an overview), SimCLR learns representations by maximizing agreement between differently augmented views of the same data example via a contrastive loss in the latent space. As illustrated in Figure 2, this framework comprises the following four major components.  \n- A stochastic _data augmentation_ module that transforms any given data example randomly resulting in two correlated views of the same example, denoted _**x**_ ˜ _i_ and _**x**_ ˜ _j_ , which we consider as a positive pair. In this work, we sequentially apply three simple augmentations: _random cropping_ followed by resize back to the original size, _random color distortions_ , and _random Gaussian blur_ . As shown in Section 3, the combination of random crop and color distortion is crucial to achieve a good performance.',
   'As we have noted in the main text, most individual compone

In [ ]:
from IPython.display import display, Markdown
display(Markdown(collection.get("7")['documents'][0]))

We combine these findings to achieve a new state-of-the-art in self-supervised and semi-supervised learning on ImageNet ILSVRC-2012 (Russakovsky et al., 2015). Under the linear evaluation protocol, SimCLR achieves 76.5% top-1 accuracy, which is a 7% relative improvement over previous state-of-the-art (Hénaff et al., 2019). When fine-tuned with only 1% of the ImageNet labels, SimCLR achieves 85.8% top-5 accuracy, a relative improvement of 10% (Hénaff et al., 2019). When fine-tuned on other natural image classification datasets, SimCLR performs on par with or better than a strong supervised baseline (Kornblith et al., 2019) on 10 out of 12 datasets.

In [ ]:
# queries = [
#     "what is SimCLR?",
#     "what data augmentation does SimCLR use?",
#     "how does the contrastive loss work?",
#     "what is the projection head?",
# ]

queries = [
    "SimCLR is a framework for contrastive learning",        # rephrased
    "what data augmentation does SimCLR use?",               # already good
    "how does the contrastive loss work?",                   # already good  
    "projection head architecture non-linear MLP",           # rephrased
]

def scoreboard(distances):
    score = 0
    weights = [1.0, 0.75, 0.55, 0,35]
    for val, weights in zip(distances[0], weights):
        if val > 1.35:
            score += -1 * weights
        elif val < 1:
            score += 1 * weights
        else:
            score += 0.5 * weights
    return round(score, 2)
        

# scores = []
for query in queries:
    results = collection.query(
        query_texts=[query],
        n_results=4,
        include=["documents", "distances"]
    )
    print(f"Query: {query}")
    print(f"Score: {scoreboard(results['distances'])}")

Query: SimCLR is a framework for contrastive learning
Score: 2.3
Query: what data augmentation does SimCLR use?
Score: 2.02
Query: how does the contrastive loss work?
Score: 2.3
Query: projection head architecture non-linear MLP
Score: 1.65


In [6]:
from huggingface_hub import login
import os
from dotenv import load_dotenv

load_dotenv(override=True)

login(os.environ['HF_TOKEN'])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [5]:
from huggingface_hub import InferenceClient

hf_client = InferenceClient()

def hyde_query(question):
    response = hf_client.chat_completion(
        model="meta-llama/Llama-3.1-8B-Instruct",
        messages=[
            {
                "role": "system",
                "content": "You are a engineering tutor assistant. Given a question, write a short factual paragraph that directly answers it, as if you were writing it in a research paper. Do not ask questions back."
            },
            {
                "role": "user",
                "content": question
            }
        ],
        max_tokens=150,
    )
    return response.choices[0].message.content

# test it
hyde_q = hyde_query("what is SimCLR in context of Machien Learning?")
print(hyde_q)

SimCLR, short for Simplified Contrastive Learning of Visual Representations, is a popular unsupervised learning algorithm in the context of machine learning, specifically computer vision. Introduced in a 2019 research paper, SimCLR is designed to learn robust and transferable representations by maximizing the agreement between two augmentations of the same image. This is achieved by contrasting two views of the same input, which are generated through random augmentations such as rotation, flipping, and color jittering. The goal is to update the model such that the agreement between the two augmented views is maximized, and the disagreement with other images is maximized, resulting in a more robust and generalizable representation of the input data. SimCLR has been widely adopted in various computer


In [ ]:
# queries = [
#     "what is SimCLR?",
#     "what data augmentation does SimCLR use?",
#     "how does the contrastive loss work?",
#     "what is the projection head?",
# ]

queries = [
    "SimCLR is a framework for contrastive learning",        # rephrased
    "what data augmentation does SimCLR use?",               # already good
    "how does the contrastive loss work?",                   # already good  
    "projection head architecture non-linear MLP",           # rephrased
]

def get_infoed_query(query):
    return query + hyde_query(query)

def rephrase_queries(queries):
    new_q = []
    for query in queries:
        new_q.append(get_infoed_query(query))
    return new_q

def scoreboard(distances):
    score = 0
    weights = [1.0, 0.75, 0.55, 0,35]
    for val, weights in zip(distances[0], weights):
        if val > 1.35:
            score += -1 * weights
        elif val < 1:
            score += 1 * weights
        else:
            score += 0.5 * weights
    return round(score, 2)
        

## Firstly let's get better queries using hyde
queries_new = rephrase_queries(queries)

for query in queries_new:    
    results = collection.query(
        query_texts=[query],
        n_results=4,
        include=["documents", "distances"]
    )
    print(f"Query: {query[:40]}...")
    print(f"Score: {scoreboard(results['distances'])}")

print("="*20)

for query in queries:    
    results = collection.query(
        query_texts=[query],
        n_results=4,
        include=["documents", "distances"]
    )
    print(f"Query: {query[:40]}...")
    print(f"Score: {scoreboard(results['distances'])}")

Query: SimCLR is a framework for contrastive le...
Score: 2.3
Query: what data augmentation does SimCLR use?S...
Score: 2.3
Query: how does the contrastive loss work?The c...
Score: 2.3
Query: projection head architecture non-linear ...
Score: 2.3
Query: SimCLR is a framework for contrastive le...
Score: 2.3
Query: what data augmentation does SimCLR use?...
Score: 2.02
Query: how does the contrastive loss work?...
Score: 2.3
Query: projection head architecture non-linear ...
Score: 1.65


In [ ]:
from pathlib import Path
import sys
sys.path.append(str(Path().resolve().parent))
import os
from pipeline.document_processor import PreprocessDocument

obj = PreprocessDocument(os.path.join(os.getcwd(), "../docs")).save_to_chroma()
response = obj.query_kb("projection head architecture non-linear MLP")


/Users/ayushjaswal/Developer/personal-projects/groundedQAproject/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading 2 files with 9 workers


2it [00:08,  4.14s/it]
Batches: 100%|██████████| 3/3 [00:00<00:00,  6.94it/s]


Stored 106 chunks from SimCLR.pdf
Stored 78 chunks from CLRMR.pdf
Total in collection: 184


In [8]:
# queries = [
#     "what is SimCLR?",
#     "what data augmentation does SimCLR use?",
#     "how does the contrastive loss work?",
#     "what is the projection head?",
# ]

queries = [
    "What is Contrastive learning for Music?",        # rephrased
    "what data augmentation does SimCLR use?",               # already good
    "how does the Proxy loss for Contrsative Learning in Music work?",                   # already good  
    "projection head architecture non-linear MLP",           # rephrased
]

def get_infoed_query(query):
    return query + hyde_query(query)

def rephrase_queries(queries):
    new_q = []
    for query in queries:
        new_q.append(get_infoed_query(query))
    return new_q

def scoreboard(distances):
    score = 0
    weights = [1.0, 0.75, 0.55, 0,35]
    for val, weights in zip(distances[0], weights):
        if val > 1.35:
            score += -1 * weights
        elif val < 1:
            score += 1 * weights
        else:
            score += 0.5 * weights
    return round(score, 2)
        

## Firstly let's get better queries using hyde
queries_new = rephrase_queries(queries)

for query in queries_new:    
    results_new = collection.query(
        query_texts=[query],
        n_results=4,
        include=["documents", "distances"]
    )
    print(f"Query: {query[:40]}...")
    print(f"Score: {scoreboard(results_new['distances'])}")

print("="*20)

for query in queries:    
    results = collection.query(
        query_texts=[query],
        n_results=4,
        include=["documents", "distances"]
    )
    print(f"Query: {query[:40]}...")
    print(f"Score: {scoreboard(results['distances'])}")

Query: What is Contrastive learning for Music?C...
Score: 2.3
Query: what data augmentation does SimCLR use?S...
Score: 2.3
Query: how does the Proxy loss for Contrsative ...
Score: 2.3
Query: projection head architecture non-linear ...
Score: 2.02
Query: What is Contrastive learning for Music?...
Score: 2.3
Query: what data augmentation does SimCLR use?...
Score: 2.02
Query: how does the Proxy loss for Contrsative ...
Score: 1.15
Query: projection head architecture non-linear ...
Score: 1.65


In [15]:
queries_new[2]

"how does the Proxy loss for Contrsative Learning in Music work?The Proxy loss for Contrastive Learning in Music is an extension of the contrastive loss function, widely used in self-supervised learning to measure the similarity between positive pairs and dissimilarity between negative pairs. The Proxy loss introduces a proxy embedding space, through a learnable neural network, where positive pairs of musical samples are mapped closer together and negative pairs are mapped far apart.\n\nThe Proxy loss is formulated as follows: let x and x' be a positive pair of musical samples, and x and z be a negative pair, where z is a different sample from a mini-batch of data. The contrastive loss function is then defined as the difference between two terms:\n\n- The infoNCE term, which encourages positive pairs to have similar embeddings\n"